# 05 — Accelerators & IP Contribution

This live lab turns a one-off Claude call into a small reusable asset, derives requirements, chooses a deployment under constraints, pins the model, maps trust boundaries, and gates promotion. Start with the [25-screen module index](../course%20content%20HTML/05-accelerators-ip-contribution/index.html).

> Running all cells requires `OPENROUTER_API_KEY` and uses paid API tokens.

## Setup

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "study_support.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

The shared setup supplies the only project-wide defaults: OpenRouter's Anthropic-compatible endpoint and one pinned Claude model.

In [ ]:
from study_support import claude_client, claude_model, message_text

client = claude_client()
MODEL = claude_model()

## Parameterize only the real seam

A reusable accelerator has explicit inputs, pinned defaults, a runnable example, and a verification check. Do not turn every constant into configuration. [Course screen 2](../course%20content%20HTML/05-accelerators-ip-contribution/02-packaging-for-reuse.html#screen-2--packaging-a-working-build-so-the-next-engagement-starts-from-an-asset)

In [ ]:
SETTINGS = {
    "model": MODEL,
    "max_tokens": 120,
    "system": "Draft concise support replies. Never invent policy or promise an outcome.",
}

The customer request is the variable seam. The safety instruction and output budget remain owned by the asset.

In [ ]:
def draft_reply(request):
    return client.messages.create(
        model=SETTINGS["model"],
        max_tokens=SETTINGS["max_tokens"],
        system=SETTINGS["system"],
        messages=[{"role": "user", "content": request}],
    )

A runnable example is more useful to the next maintainer than a claim that the template is reusable.

In [ ]:
example = draft_reply("I was charged twice. Acknowledge the issue and explain the next step.")
example_text = message_text(example)
print(example_text)

## Extract functional and infrastructure requirements

Functional requirements describe behavior. Infrastructure requirements cover latency, region, identity, scale, data handling, and operations. Record both before choosing a platform. [Course screen 8](../course%20content%20HTML/05-accelerators-ip-contribution/04-requirements-lifecycle.html#screen-8--from-business-requirements-to-functional-and-infrastructure-requirements)

In [ ]:
business_need = (
    "Draft cited replies to EU customer emails in under two seconds. "
    "A human must approve every send, and the service uses a read-only identity."
)

Claude proposes a structured extraction; application code validates the keys because model output remains untrusted input.

In [ ]:
requirements_call = client.messages.create(
    model=MODEL,
    max_tokens=180,
    messages=[{"role": "user", "content": (
        "Return only JSON with arrays functional and infrastructure. " + business_need
    )}],
)
raw_requirements = message_text(requirements_call)

Reject malformed or incomplete output before using it in a design decision.

In [ ]:
import json

clean_json = raw_requirements.strip().removeprefix("```json").removesuffix("```").strip()
try:
    requirements = json.loads(clean_json)
except json.JSONDecodeError:
    requirements = {}
requirements_valid = set(requirements) == {"functional", "infrastructure"}
{"valid": requirements_valid, "value": requirements, "raw": raw_requirements}

## Gate each lifecycle phase

Discovery informs design; eval evidence gates deployment; monitoring and rollback belong to operation. [Course screen 10](../course%20content%20HTML/05-accelerators-ip-contribution/04-requirements-lifecycle.html#screen-10--systems-lifecycle-for-claude-applications)

In [ ]:
gates = {
    "design": "requirements approved",
    "evaluate": "quality, latency, and safety thresholds met",
    "deploy": "human approval",
    "operate": "traces, alerts, and rollback ready",
}
assert all(gates.values())

## Choose the platform from constraints

Compliance can eliminate a platform before latency and token price are compared. Measure from the customer's region and include operational cost. The values below are fictional scenario inputs, not current provider claims. [Course screens 12 and 15](../course%20content%20HTML/05-accelerators-ip-contribution/05-deployment-versioning.html#screen-12--choosing-where-a-claude-workload-runs-and-versioning-what-ships)

In [ ]:
platforms = [
    {"name": "first-party", "eu_residency": False, "latency_ms": 700},
    {"name": "bedrock-eu", "eu_residency": True, "latency_ms": 850},
]
eligible = [platform for platform in platforms if platform["eu_residency"]]
choice = min(eligible, key=lambda platform: platform["latency_ms"])
choice

This notebook reaches Claude through OpenRouter. The platform comparison is a design exercise; changing `base_url` alone does not turn this client into a Bedrock client.

## Pin what ships

The default `anthropic/claude-sonnet-4.6` is explicit in `.env.example`. Change `CLAUDE_MODEL` only as a deliberate migration and rerun the bundled eval before promotion. [Course screen 13](../course%20content%20HTML/05-accelerators-ip-contribution/05-deployment-versioning.html#screen-13--the-deployment-that-broke-when-the-model-alias-moved)

In [ ]:
release = {
    "model": SETTINGS["model"],
    "prompt_version": "support-reply-v1",
    "previous_model_retained": True,
}
release

## Map trust boundaries before connecting components

Every seam that moves data is a trust boundary. The most privileged component needs the narrowest identity and the strongest audit controls. [Course screen 18](../course%20content%20HTML/05-accelerators-ip-contribution/07-trust-boundaries.html#screen-18--coordinating-several-claude-deployments-with-the-trust-boundaries-holding-under-review)

In [ ]:
boundaries = {
    "email_to_api": {"trust": "untrusted", "control": "isolate as data"},
    "api_to_mcp": {"identity": "svc-readonly", "permissions": ["customer:read"]},
    "draft_to_send": {"control": "human approval"},
}

Use Claude to look for omissions, but do not let the review grant itself additional access.

In [ ]:
boundary_review = client.messages.create(
    model=MODEL,
    max_tokens=140,
    system="Review trust boundaries for least privilege. List omissions only.",
    messages=[{"role": "user", "content": json.dumps(boundaries)}],
)
print(message_text(boundary_review))

## Verify before promotion

A runnable example, test evidence, documented assumptions, clear rights, and a rollback path make an asset reviewable. Human approval remains a real gate.

In [ ]:
readiness = {
    "example_runs": bool(example_text),
    "requirements_recorded": requirements_valid,
    "model_pinned": SETTINGS["model"] == MODEL,
    "rollback_ready": release["previous_model_retained"],
}
human_approved = False
promotion = "promote" if all(readiness.values()) and human_approved else "hold"
promotion

## Try it

Add one real requirement from your intended deployment. Update the platform filter, boundary map, and readiness gate so that requirement can be defended from design through operation.